# Papapanagiotou Panagiotis

### Εκφώνηση: γράψτε ένα πλήρες PCTF prompt για το παρακάτω σενάριο:
 - ####  Writing test cases for a function. 
    - Να δημιουργήσετε ένα prompt που ζητά από το μοντέλο να γράψει test cases για μια συνάρτηση
(π.χ. unit tests, edge cases, boundary values).

#### My basic setup code

In [5]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output
 
load_dotenv(Path.cwd().parent / "../module_02_prompt_enginnering/.env")

from anthropic import Anthropic

claude_client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
CLAUDE_MODEL = "claude-opus-4-6" 

if claude_client:
    print(f"Antropic client ready - using model {CLAUDE_MODEL}")

Antropic client ready - using model claude-opus-4-6


#### Η συνάρτηση που καλεί το API της Athropic

In [2]:
def ask_anthropic(prompt: str, system: str | None = None, temperature: float= 0.7, max_resp_tokens: int= 800, ai_model: str= "claude-sonnet-4-6", output_format: str | None = None) -> None:
    """
    Send a prompt to Anthropic using the streaming Messages API and return
    the complete generated text.

    Args:
        prompt (str):
            The user prompt to send to the model.
        system (str | None, optional):
            Optional system instruction for the model. In the streaming API
            for this SDK version, it is converted into the required list format.
            Defaults to None.
        temperature (float, optional):
            Controls randomness of the response. Defaults to 0.7.
        max_resp_tokens (int, optional):
            Maximum number of tokens in the generated response. Defaults to 800.
        ai_model (str, optional):
            The model name to use. Defaults to "claude-sonnet-4-6".
        output_format (str | None, optional):
            Optional output format for the streaming API, if supported by the
            installed SDK version. Defaults to None.

    Returns: None
    """

    params = {
        "model": ai_model,
        "max_tokens" : max_resp_tokens,
        "temperature": temperature,
        "messages": [
            {'role': 'user', 'content': prompt}
        ],
    }

    if system:
        params["system"] = [{"type": "text", "text": system}]

    if output_format:
        params["output_format"] = output_format
    
    with claude_client.messages.stream(**params) as stream:
        full_text = ""
        for text in stream.text_stream:
            full_text += text
            clear_output(wait=True)
            display(Markdown(full_text))
    
    #return full_text

display(Markdown(f"**The function has run**"))

**The function has run**

### Η PCTF συνάρτηση που κάναμε στο μάθημα:

In [4]:
def create_pctf_prompt(persona: str, context: str, task: str, format_spec: str) -> str:
    """Build a structred PCTF prompt"""
    return f"""## Persona
{persona}

## Context
{context}

## Task
{task}

## Format
{format_spec}
"""

display(Markdown("#### The function has run"))

#### The function has run

In [6]:
persona = "Are you an experienced software developer that specialized in code testing sector."

context = "We have to test many lines of Java code for our new software developer project in banking sector"

task = """Write a unit test, check the edge cases if exists, the boundary values and the error messages of this class in Java:
public class BankAccount {
    
    public static String processWithdrawal(double balance, double amount, boolean isPremiumCustomer) {
        // Validate inputs
        if (balance < 0) {
            return "Error: Invalid account balance";
        }
        if (amount <= 0) {
            return "Error: Invalid withdrawal amount";
        }
        
        // Check if sufficient funds
        if (amount > balance) {
            // Premium customers can overdraw up to $100
            if (isPremiumCustomer && amount <= balance + 100) {
                return "Warning: Overdraft applied. New balance: $" + 
                       String.format("%.2f", balance - amount);
            }
            return "Error: Insufficient funds";
        }
        
        // Regular withdrawal
        double newBalance = balance - amount;
        
        // Alert if balance falls below $50
        if (newBalance < 50) {
            return "Warning: Low balance. New balance: $" + 
                   String.format("%.2f", newBalance);
        }
        
        return "Success: New balance: $" + String.format("%.2f", newBalance);
    }
}
"""
format_spec = "Return ONLY the code that will use for the testing with a small comment for the purpose of use."

display(Markdown("#### PCTF values has load"))

#### PCTF values has load

In [9]:
prompt = create_pctf_prompt(persona, context, task, format_spec)

ask_anthropic(prompt, temperature= 0.0, max_resp_tokens= 5000, ai_model= CLAUDE_MODEL)



```java
import org.junit.jupiter.api.Test;
import org.junit.jupiter.api.DisplayName;
import org.junit.jupiter.params.ParameterizedTest;
import org.junit.jupiter.params.provider.CsvSource;

import static org.junit.jupiter.api.Assertions.*;

/**
 * Unit tests for BankAccount.processWithdrawal()
 * Covers: input validation, sufficient/insufficient funds, premium overdraft,
 * low balance warnings, boundary values, and edge cases.
 */
class BankAccountTest {

    // ==================== INPUT VALIDATION TESTS ====================

    @Test
    @DisplayName("Negative balance should return invalid account balance error")
    void negativeBalance_returnsInvalidBalanceError() {
        assertEquals("Error: Invalid account balance",
                BankAccount.processWithdrawal(-1.0, 50.0, false));
    }

    @Test
    @DisplayName("Negative balance with premium customer should still return invalid balance error")
    void negativeBalance_premiumCustomer_returnsInvalidBalanceError() {
        assertEquals("Error: Invalid account balance",
                BankAccount.processWithdrawal(-0.01, 10.0, true));
    }

    @Test
    @DisplayName("Zero withdrawal amount should return invalid withdrawal amount error")
    void zeroAmount_returnsInvalidAmountError() {
        assertEquals("Error: Invalid withdrawal amount",
                BankAccount.processWithdrawal(100.0, 0.0, false));
    }

    @Test
    @DisplayName("Negative withdrawal amount should return invalid withdrawal amount error")
    void negativeAmount_returnsInvalidAmountError() {
        assertEquals("Error: Invalid withdrawal amount",
                BankAccount.processWithdrawal(100.0, -50.0, false));
    }

    @Test
    @DisplayName("Both balance negative and amount zero — balance check takes priority")
    void negativeBalanceAndZeroAmount_balanceErrorFirst() {
        // Validates that balance is checked before amount
        assertEquals("Error: Invalid account balance",
                BankAccount.processWithdrawal(-5.0, 0.0, false));
    }

    @Test
    @DisplayName("Both balance negative and amount negative — balance check takes priority")
    void negativeBalanceAndNegativeAmount_balanceErrorFirst() {
        assertEquals("Error: Invalid account balance",
                BankAccount.processWithdrawal(-1.0, -1.0, true));
    }

    // ==================== SUCCESSFUL WITHDRAWAL TESTS ====================

    @Test
    @DisplayName("Regular withdrawal with sufficient funds and balance stays above $50")
    void regularWithdrawal_sufficientFunds_aboveThreshold() {
        assertEquals("Success: New balance: $150.00",
                BankAccount.processWithdrawal(200.0, 50.0, false));
    }

    @Test
    @DisplayName("Premium customer regular withdrawal with sufficient funds")
    void premiumCustomer_regularWithdrawal_sufficientFunds() {
        assertEquals("Success: New balance: $500.00",
                BankAccount.processWithdrawal(1000.0, 500.0, true));
    }

    @Test
    @DisplayName("Withdraw entire balance (result = $0.00) triggers low balance warning")
    void withdrawEntireBalance_lowBalanceWarning() {
        assertEquals("Warning: Low balance. New balance: $0.00",
                BankAccount.processWithdrawal(100.0, 100.0, false));
    }

    // ==================== LOW BALANCE WARNING TESTS ====================

    @Test
    @DisplayName("New balance exactly $49.99 triggers low balance warning")
    void newBalanceJustBelow50_lowBalanceWarning() {
        assertEquals("Warning: Low balance. New balance: $49.99",
                BankAccount.processWithdrawal(100.0, 50.01, false));
    }

    @Test
    @DisplayName("New balance exactly $50.00 does NOT trigger low balance warning")
    void newBalanceExactly50_noWarning() {
        assertEquals("Success: New balance: $50.00",
                BankAccount.processWithdrawal(100.0, 50.0, false));
    }

    @Test
    @DisplayName("New balance $50.01 does NOT trigger low balance warning")
    void newBalanceJustAbove50_noWarning() {
        assertEquals("Success: New balance: $50.01",
                BankAccount.processWithdrawal(100.0, 49.99, false));
    }

    @Test
    @DisplayName("New balance $0.01 triggers low balance warning")
    void newBalanceVerySmall_lowBalanceWarning() {
        assertEquals("Warning: Low balance. New balance: $0.01",
                BankAccount.processWithdrawal(50.0, 49.99, false));
    }

    // ==================== INSUFFICIENT FUNDS TESTS ====================

    @Test
    @DisplayName("Non-premium customer with insufficient funds returns error")
    void nonPremium_insufficientFunds_returnsError() {
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(50.0, 100.0, false));
    }

    @Test
    @DisplayName("Non-premium customer overdraft within $100 still returns insufficient funds")
    void nonPremium_overdraftWithin100_returnsError() {
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(50.0, 60.0, false));
    }

    @Test
    @DisplayName("Non-premium customer amount exceeds balance by exactly $100 returns error")
    void nonPremium_overdraftExactly100_returnsError() {
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(0.0, 100.0, false));
    }

    // ==================== PREMIUM CUSTOMER OVERDRAFT TESTS ====================

    @Test
    @DisplayName("Premium customer overdraft within $100 limit applies overdraft")
    void premium_overdraftWithinLimit_appliesOverdraft() {
        assertEquals("Warning: Overdraft applied. New balance: $-50.00",
                BankAccount.processWithdrawal(50.0, 100.0, true));
    }

    @Test
    @DisplayName("Premium customer overdraft exactly at $100 limit boundary")
    void premium_overdraftExactlyAtLimit_appliesOverdraft() {
        // amount = balance + 100 => 200 = 100 + 100, should apply overdraft
        assertEquals("Warning: Overdraft applied. New balance: $-100.00",
                BankAccount.processWithdrawal(100.0, 200.0, true));
    }

    @Test
    @DisplayName("Premium customer overdraft exceeds $100 limit by $0.01 returns insufficient funds")
    void premium_overdraftExceedsLimitByPenny_returnsError() {
        // amount = balance + 100.01 => exceeds limit
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(100.0, 200.01, true));
    }

    @Test
    @DisplayName("Premium customer overdraft exceeds $100 limit significantly returns insufficient funds")
    void premium_overdraftFarExceedsLimit_returnsError() {
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(50.0, 500.0, true));
    }

    @Test
    @DisplayName("Premium customer with zero balance can overdraft up to $100")
    void premium_zeroBalance_overdraftWithinLimit() {
        assertEquals("Warning: Overdraft applied. New balance: $-99.99",
                BankAccount.processWithdrawal(0.0, 99.99, true));
    }

    @Test
    @DisplayName("Premium customer with zero balance overdraft exactly $100")
    void premium_zeroBalance_overdraftExactly100() {
        assertEquals("Warning: Overdraft applied. New balance: $-100.00",
                BankAccount.processWithdrawal(0.0, 100.0, true));
    }

    @Test
    @DisplayName("Premium customer with zero balance overdraft $100.01 returns error")
    void premium_zeroBalance_overdraftExceeds100() {
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(0.0, 100.01, true));
    }

    @Test
    @DisplayName("Premium customer overdraft by $0.01 applies overdraft")
    void premium_overdraftByOnePenny() {
        assertEquals("Warning: Overdraft applied. New balance: $-0.01",
                BankAccount.processWithdrawal(100.0, 100.01, true));
    }

    // ==================== BOUNDARY VALUE: BALANCE = 0 ====================

    @Test
    @DisplayName("Zero balance with any positive withdrawal for non-premium returns insufficient funds")
    void zeroBalance_nonPremium_insufficientFunds() {
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(0.0, 0.01, false));
    }

    @Test
    @DisplayName("Zero balance with small withdrawal for premium applies overdraft")
    void zeroBalance_premium_smallWithdrawal_overdraft() {
        assertEquals("Warning: Overdraft applied. New balance: $-0.01",
                BankAccount.processWithdrawal(0.0, 0.01, true));
    }

    // ==================== BOUNDARY VALUE: AMOUNT = BALANCE ====================

    @Test
    @DisplayName("Amount exactly equals balance — no overdraft needed, low balance warning")
    void amountEqualsBalance_lowBalanceWarning() {
        assertEquals("Warning: Low balance. New balance: $0.00",
                BankAccount.processWithdrawal(30.0, 30.0, false));
    }

    @Test
    @DisplayName("Amount exactly equals balance above $50 — success")
    void amountEqualsBalance_above50_noWarning() {
        // balance = 50, amount = 50, newBalance = 0 => low balance
        assertEquals("Warning: Low balance. New balance: $0.00",
                BankAccount.processWithdrawal(50.0, 50.0, true));
    }

    // ==================== EDGE CASES: VERY SMALL VALUES ====================

    @Test
    @DisplayName("Very small positive amount withdrawal from large balance")
    void verySmallAmount_largeBalance() {
        assertEquals("Success: New balance: $999999.99",
                BankAccount.processWithdrawal(1000000.0, 0.01, false));
    }

    @Test
    @DisplayName("Very small balance boundary — balance just above zero")
    void verySmallBalance_withdrawExact() {
        assertEquals("Warning: Low balance. New balance: $0.00",
                BankAccount.processWithdrawal(0.01, 0.01, false));
    }

    // ==================== EDGE CASES: LARGE VALUES ====================

    @Test
    @DisplayName("Large balance and large withdrawal — success")
    void largeValues_success() {
        assertEquals("Success: New balance: $500000.00",
                BankAccount.processWithdrawal(1000000.0, 500000.0, false));
    }

    @Test
    @DisplayName("Large balance and large withdrawal — insufficient funds")
    void largeValues_insufficientFunds() {
        assertEquals("Error: Insufficient funds",
                BankAccount.processWithdrawal(1000000.0, 2000000.0, false));
    }

    // ==================== EDGE CASE: BALANCE EXACTLY 0 (NOT NEGATIVE) ====================

    @Test
    @DisplayName("Balance exactly 0 is valid — not an error")
    void zeroBalance_isValid() {
        // Should NOT return "Error: Invalid account balance"
        String result = BankAccount.processWithdrawal(0.0, 50.0, false);
        assertNotEquals("Error: Invalid account balance", result);
    }

    // ==================== PARAMETERIZED TESTS FOR COMPREHENSIVE COVERAGE ====================

    @ParameterizedTest(name = "balance={0}, amount={1}, premium={2} => {3}")
    @CsvSource({
            // Regular successful withdrawals
            "200.00, 50.00, false, Success: New balance: $150.00",
            "1000.00, 100.00, true, Success: New balance: $900.00",
            // Low balance warnings
            "60.00, 20.00, false, Warning: Low balance. New balance: $40.00",
            "49.99, 10.00, true, Warning: Low balance. New balance: $39.99",
            // Overdraft for premium
            "10.00, 50.00, true, Warning: Overdraft applied. New balance: $-40.00",
            "0.00, 100.00, true, Warning: Overdraft applied. New balance: $-100.00",
            // Insufficient funds
            "10.00, 50.00, false, Error: Insufficient funds",
            "100.00, 300.00, true, Error: Insufficient funds",
    })
    void parameterizedWithdrawalTests(double balance, double amount, boolean isPremium, String expected) {
        assertEquals(expected, BankAccount.processWithdrawal(balance, amount, isPremium));
    }

    // ==================== FORMATTING VERIFICATION ====================

    @Test
    @DisplayName("Verify decimal formatting to two places for whole number result")
    void formattingWholeNumber() {
        String result = BankAccount.processWithdrawal(200.0, 100.0, false);
        assertTrue(result.contains("$100.00"), "Should format to two decimal places");
    }

    @Test
    @DisplayName("Verify decimal formatting for overdraft negative result")
    void formattingNegativeOverdraft() {
        String result = BankAccount.processWithdrawal(10.0, 60.0, true);
        assertTrue(result.contains("$-50.00"), "Negative overdraft should be formatted correctly");
    }

    @Test
    @DisplayName("Verify message prefix for success scenario")
    void successMessagePrefix() {
        String result = BankAccount.processWithdrawal(500.0, 100.0, false);
        assertTrue(result.startsWith("Success:"), "Should start with 'Success:'");
    }

    @Test
    @DisplayName("Verify message prefix for low balance warning")
    void lowBalanceWarningPrefix() {
        String result = BankAccount.processWithdrawal(60.0, 20.0, false);
        assertTrue(result.startsWith("Warning: Low balance."), "Should start with 'Warning: Low balance.'");
    }

    @Test
    @DisplayName("Verify message prefix for overdraft warning")
    void overdraftWarningPrefix() {
        String result = BankAccount.processWithdrawal(10.0, 50.0, true);
        assertTrue(result.startsWith("Warning: Overdraft applied."), "Should start with 'Warning: Overdraft applied.'");
    }
}
```